# 🔧 StreamCI Advanced — PyStreamCI Client Library

This notebook covers **advanced features** of the `pystreamci` client library:

1. **Streaming Query** — process large result sets without loading everything into memory
2. **Update** — modify existing documents
3. **Delete** — remove documents
4. **Blob Operations** — insert, query, and download binary files (images, etc.)

> 📋 **Prerequisite:** Complete `new-01-quickstart.ipynb` first to populate data in your StreamCI target.

In [ ]:
import pystreamci

# ✏️ Fill in your credentials (same as Quickstart)
HOST       = "https://api.streamci.org"
TARGET     = "userXX"
SECRET_KEY = ""

---
## 1. Streaming Query (`query_iter`)

For **large result sets**, `client.query_iter()` returns an **iterator** that streams results one document at a time using NDJSON over HTTP. This keeps memory usage constant, regardless of result size.

| Method | Returns | Best for |
|--------|---------|----------|
| `query()` | `list[dict]` | Small results that fit in memory |
| `query_iter()` | `Iterator[dict]` | Large datasets; process one doc at a time |

In [ ]:
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:
    count = 0
    for doc in client.query_iter({}):
        count += 1
        if count <= 3:
            print(f"  Doc {count}: {doc}")

    print(f"\nTotal documents streamed: {count}")

In [ ]:
# Example: stream with filters (same parameters as query())
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:
    sensor1_readings = []
    for doc in client.query_iter(
        {"deviceID": "sensor1"},
        project=["time", "val1", "val2"],
        sort={"time": 1},
    ):
        sensor1_readings.append(doc)

    print(f"Streamed {len(sensor1_readings)} sensor1 records")
    if sensor1_readings:
        print("First:", sensor1_readings[0])
        print("Last: ", sensor1_readings[-1])

---
## 2. Update Documents

`client.update(filter, new_data)` modifies documents matching the filter.

Let's:
1. Query a specific record
2. Update its value
3. Query again to verify the change

In [ ]:
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:

    # Step 1: Find a record to update
    before = client.query(
        {"deviceID": "sensor1"},
        project=["time", "deviceID", "val1", "val2"],
        sort={"time": 1},
        limit=1,
    )
    print("Before update:", before[0] if before else "No records found")

In [ ]:
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:

    # Step 2: Update — change val1 to 999 for the earliest sensor1 record
    result = client.update(
        {"deviceID": "sensor1", "time": before[0]["time"]},   # filter
        {"val1": 999},                                         # new data
    )
    print("Update result:", result)

In [ ]:
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:

    # Step 3: Verify the update
    after = client.query(
        {"deviceID": "sensor1", "val1": 999},
        project=["time", "deviceID", "val1", "val2"],
        limit=1,
    )
    print("After update:", after[0] if after else "Not found")

---
## 3. Delete Documents

`client.delete(filter)` removes all documents matching the filter.

> ⚠️ **Caution:** Deletion is permanent. Always double-check your filter before running.

In [ ]:
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:

    # Delete the record we just updated (val1 == 999)
    result = client.delete({"deviceID": "sensor1", "val1": 999})
    print("Delete result:", result)

In [ ]:
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:

    # Verify deletion
    check = client.query({"deviceID": "sensor1", "val1": 999})
    print(f"Records with val1=999: {len(check)} (should be 0)")

---
## 4. Blob Operations

StreamCI supports **binary large objects (blobs)** — such as images, PDFs, or any binary files — attached to documents.

### 4-1. Insert a Document with Blob Attachments

Use the `blobs` parameter to attach binary files when inserting a document.  
The `blobs` dict accepts several convenient formats:

| Format | Example |
|--------|---------|
| File path `str` / `Path` | `"photo.jpg"` or `Path("photo.jpg")` |
| File object (from `open()`) | `f` |
| 2-tuple `(filename, file_obj)` | `("photo.jpg", f)` |
| 3-tuple `(filename, file_obj, mime)` | `("photo.jpg", f, "image/jpeg")` |

Filename and MIME type are **auto-inferred** when not provided.

In [ ]:
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:

    # Simplest: just pass file paths — filename and MIME type are auto-inferred
    result = client.insert(
        {
            "time": pystreamci.make_date("2025-08-04T10:00:00.001Z"),
            "lat":  40.429,
            "lon":  -86.917,
            "tag":  "Purdue",
        },
        blobs={
            "image1": "example-image1.jpg",
            "image2": "example-image2.jpg",
        },
    )
    print("Insert with blobs:", result)

### 4-2. Query Records and Download Blobs

First, query the record to get its `_id`, then use `client.query_blob()` to download the binary data.

In [ ]:
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:

    # Find the record with blobs
    docs = client.query(
        {"tag": "Purdue"},
        project=["_id", "tag", "image1", "image2"],
    )
    print("Found:", docs[0] if docs else "No records")

    if docs:
        doc_id = docs[0]["_id"]

        # Download each blob
        img1_bytes = client.query_blob(doc_id, "image1")
        img2_bytes = client.query_blob(doc_id, "image2")
        print(f"\nDownloaded image1: {len(img1_bytes)} bytes")
        print(f"Downloaded image2: {len(img2_bytes)} bytes")

In [ ]:
# Display the downloaded images in the notebook
from IPython.display import Image, display

print("Image 1:")
display(Image(data=img1_bytes))

print("Image 2:")
display(Image(data=img2_bytes))

### 4-3. Download Multiple Blobs at Once

`client.query_blobs_by_field()` downloads the same field from **multiple documents** in one call.

In [ ]:
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:

    # If you have multiple documents with blob fields:
    docs = client.query({"tag": "Purdue"}, project=["_id"])
    doc_ids = [d["_id"] for d in docs]

    if doc_ids:
        blobs_map = client.query_blobs_by_field(doc_ids, "image1")
        for did, blob_bytes in blobs_map.items():
            print(f"  Document {did}: {len(blob_bytes)} bytes")

### 4-4. Cleanup — Delete Blob Records

In [ ]:
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:

    result = client.delete({"tag": "Purdue"})
    print("Deleted blob records:", result)

    # Verify
    check = client.query({"tag": "Purdue"})
    print(f"Remaining Purdue records: {len(check)} (should be 0)")

---
## 5. Nested Blob Fields

StreamCI allows blob fields to be stored at **nested paths** within a document. You specify the path using **bracket notation** (`key["subkey"]`) or **dot notation** (`key.subkey`) when inserting; the server stores them as nested objects. When downloading, always use the equivalent **dot-path**.

| Insert key | Stored as | Download path |
|------------|-----------|---------------|
| `primaryDocument["file"]` | `doc["primaryDocument"]["file"]` | `"primaryDocument.file"` |
| `attachment.file` | `doc["attachment"]["file"]` | `"attachment.file"` |
| `result["fit_result"]["figure"]` | `doc["result"]["fit_result"]["figure"]` | `"result.fit_result.figure"` |
| `figures[0]` | `doc["figures"][0]` | `"figures.0"` |

In [ ]:
# 5-1. Insert nested blob — bracket notation
from io import BytesIO

bracket_content = b"Nested blob - bracket notation example."

with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:
    result = client.insert(
        {"label": "nested-blob-bracket"},
        blobs={
            # 2-tuple: (filename, file_obj) — MIME auto-inferred from extension
            'primaryDocument["file"]': ("report.bin", BytesIO(bracket_content)),
        },
    )
    print("Insert result:", result)

In [ ]:
# 5-2. Query and download nested blob
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:
    docs = client.query({"label": "nested-blob-bracket"})
    print("Document:", docs[0] if docs else "Not found")

    if docs:
        doc_id = docs[0]["_id"]
        # Bracket notation key "primaryDocument[file]" → dot-path "primaryDocument.file"
        blob_bytes = client.query_blob(doc_id, "primaryDocument.file")
        print(f"Downloaded {len(blob_bytes)} bytes")
        assert blob_bytes == bracket_content, "Content mismatch!"
        print("Content verified ✓")

In [ ]:
# 5-3. Insert with deep nesting and array indices
deep_content = b"Deep nested blob - 2 levels."
arr_blob_0 = b"Array blob index 0."
arr_blob_1 = b"Array blob index 1."

with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:
    # Deep nesting: result["fit_result"]["figure"]
    r1 = client.insert(
        {"label": "deep-nested-blob"},
        blobs={
            'result["fit_result"]["figure"]': ("figure.bin", BytesIO(deep_content)),
        },
    )
    print("Deep nested insert:", r1)

    # Array indices: figures[0], figures[1]
    r2 = client.insert(
        {"label": "array-blob-doc"},
        blobs={
            "figures[0]": ("fig0.bin", BytesIO(arr_blob_0)),
            "figures[1]": ("fig1.bin", BytesIO(arr_blob_1)),
        },
    )
    print("Array blobs insert:", r2)

In [ ]:
# 5-4. Download deep nested and array-indexed blobs
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:
    # Deep nested: download via dot-path result.fit_result.figure
    docs = client.query({"label": "deep-nested-blob"})
    if docs:
        doc_id = docs[0]["_id"]
        deep_bytes = client.query_blob(doc_id, "result.fit_result.figure")
        print(f"Deep nested blob: {len(deep_bytes)} bytes")
        assert deep_bytes == deep_content, "Content mismatch!"
        print("Deep nested content verified ✓")

    # Array indices: download via figures.0 and figures.1
    docs2 = client.query({"label": "array-blob-doc"})
    if docs2:
        doc_id2 = docs2[0]["_id"]
        fig0 = client.query_blob(doc_id2, "figures.0")
        fig1 = client.query_blob(doc_id2, "figures.1")
        print(f"figures.0: {len(fig0)} bytes, figures.1: {len(fig1)} bytes")
        assert fig0 == arr_blob_0 and fig1 == arr_blob_1, "Array blob content mismatch!"
        print("Array blob content verified ✓")

In [ ]:
# 5-5. Cleanup — delete nested blob documents
with pystreamci.connect(HOST, TARGET, secret_key=SECRET_KEY) as client:
    for label in ["nested-blob-bracket", "deep-nested-blob", "array-blob-doc"]:
        result = client.delete({"label": label})
        print(f"Deleted '{label}':", result)

---
## Summary

| Feature | Method |
|---------|--------|
| Stream large queries | `client.query_iter(query, project=..., sort=..., limit=...)` |
| Update documents | `client.update(filter, new_data)` |
| Delete documents | `client.delete(filter)` |
| Insert with blobs | `client.insert(data, blobs={...})` |
| Download blob | `client.query_blob(doc_id, field)` |
| Download nested blob | `client.query_blob(doc_id, "key.subkey")` |
| Download array blob | `client.query_blob(doc_id, "key.0")` |
| Download multiple blobs | `client.query_blobs_by_field([doc_ids], field)` |
| Download all blobs from doc (incl. nested) | `client.query_blobs_by_doc(doc_id)` |

For a complete API reference, see the [PyStreamCI README](../../python/README.md).